# Alpaca Weekly Backtest Notebook

This notebook runs the new Alpaca weekly-news backtest workflow end to end:

- load the weekly Alpaca config
- fetch/cache weekly news windows per symbol
- build the backtest dataset
- inspect `no_signal` / `no_article_provided` rows
- optionally run the existing LLM backtester and summarize metrics


In [ ]:
!pip install -q datasets transformers accelerate pydantic python-dotenv yfinance tqdm
!pip install -q vllm
print("Dependencies installed.")

In [ ]:
import os
from google.colab import drive
import sys

drive.mount('/content/drive')

# ── Edit these two lines to match your Drive layout ──────────────────────────
PROJECT_PATH = '/content/drive/MyDrive/FinGPT_Part2'
REPO_URL     = 'https://github.com/juankim834/FinGPT_Project.git'
# ─────────────────────────────────────────────────────────────────────────────

if os.path.isdir(os.path.join(PROJECT_PATH, '.git')):
    print(f"Repo found at {PROJECT_PATH} — pulling latest changes.")
    %cd {PROJECT_PATH}
    !git pull
else:
    print(f"Repo not found — cloning into {PROJECT_PATH}.")
    parent = os.path.dirname(PROJECT_PATH)
    os.makedirs(parent, exist_ok=True)
    %cd {parent}
    !git clone {REPO_URL} {os.path.basename(PROJECT_PATH)}
    %cd {PROJECT_PATH}

# Ensure project root is on sys.path so all `from agent1…` imports resolve.
if PROJECT_PATH not in sys.path:
    sys.path.insert(0, PROJECT_PATH)

# Shared directories persisted on Drive.
DEMO_OUTPUT_DIR = os.path.join(PROJECT_PATH, 'output')
DIAG_MD_DIR     = os.path.join(DEMO_OUTPUT_DIR, 'diagnostics_md')
for d in [DEMO_OUTPUT_DIR, DIAG_MD_DIR]:
    os.makedirs(d, exist_ok=True)

print(f"Project path : {PROJECT_PATH}")
print(f"Output dir   : {DEMO_OUTPUT_DIR}")
print(f"Diagnostics  : {DIAG_MD_DIR}")

In [ ]:
import os
from google.colab import userdata

# ── vLLM Colab / Jupyter compatibility fix ────────────────────────────────────
# vLLM ≥0.6 (v1 engine) spawns an EngineCore subprocess that calls
# sys.stdout.fileno().  IPython's OutStream has no real fileno(), so the child
# crashes with:  io.UnsupportedOperation: fileno
#
# Setting VLLM_ENABLE_V1_MULTIPROCESSING=0 forces the v1 engine to run
# in-process (InprocClient) — no subprocess spawned, no fileno call.
# Must be set BEFORE LLM() is called (vLLM reads it lazily at init time).
os.environ['VLLM_ENABLE_V1_MULTIPROCESSING'] = '0'

# ── News provider ─────────────────────────────────────────────────────────────
os.environ['NEWS_PROVIDER']   = 'finnhub'          # 'finnhub' or 'alpaca'
os.environ['FINNHUB_API_KEY'] = userdata.get('FINNHUB_API_KEY')
os.environ['ALPACA_API_KEY']  = userdata.get('ALPACA_API_KEY')
os.environ['ALPACA_API_SECRET'] = userdata.get('ALPACA_API_SECRET')

# ── Model path ────────────────────────────────────────────────────────────────
# Point to your local merged / quantised model directory on Drive.
# The folder must contain config.json (HuggingFace format).
model_candidates = [
    '/content/drive/MyDrive/deepseek_fingpt_outputs/merged_for_vllm',
    '/content/drive/MyDrive/models/DeepSeek-R1-Distill-Llama-8B',
]
resolved_model = next(
    (p for p in model_candidates if os.path.isfile(os.path.join(p, 'config.json'))),
    None,
)
if resolved_model is None:
    raise FileNotFoundError(
        'Could not find a model folder with config.json. '
        f'Checked: {model_candidates}'
    )
os.environ['FINGPT_MODEL_PATH'] = resolved_model

# ── Pipeline settings ─────────────────────────────────────────────────────────
# Share one vLLM engine between Agent 1 and Agent 2 to save VRAM.
os.environ['SHARE_SINGLE_LLM_BETWEEN_AGENTS'] = 'true'

# Calibration temperature applied to softmax(logits / T).
# T > 1 → softer distribution  |  T < 1 → sharper  |  T = 1 → standard softmax
os.environ['FINGPT_CALIBRATION_T']    = '1.2'

# Token budget for CoT + logits generation per article.
os.environ['FINGPT_LOGITS_MAX_TOKENS'] = '1024'

# Debug output directory (raw model outputs saved here for inspection).
os.environ['FINGPT_DIAG_MD_DIR'] = DIAG_MD_DIR

# Persist yfinance price cache across runs.
os.environ['FINGPT_YF_CACHE_PATH'] = os.path.join(DEMO_OUTPUT_DIR, 'yfinance_return_cache.json')

print('Environment configured.')
print(f"  NEWS_PROVIDER            = {os.environ['NEWS_PROVIDER']}")
print(f"  FINGPT_MODEL_PATH        = {os.environ['FINGPT_MODEL_PATH']}")
print(f"  FINGPT_CALIBRATION_T     = {os.environ['FINGPT_CALIBRATION_T']}")
print(f"  FINGPT_LOGITS_MAX_TOKENS = {os.environ['FINGPT_LOGITS_MAX_TOKENS']}")
print(f"  FINGPT_DIAG_MD_DIR       = {os.environ['FINGPT_DIAG_MD_DIR']}")

In [ ]:
# Cell 1 - Imports and repo path
from pathlib import Path
import json

import pandas as pd

REPO_DIR = Path.cwd()
if not (REPO_DIR / 'backtest').exists():
    raise RuntimeError(f'Run this notebook from the repo root. Current dir: {REPO_DIR}')

if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

print(f'Repo dir: {REPO_DIR}')


In [ ]:
# Cell 2 - Load config
from backtest.alpaca_news_pipeline import load_alpaca_backtest_config

CONFIG_PATH = REPO_DIR / 'configs' / 'alpaca_backtest.example.json'
config = load_alpaca_backtest_config(str(CONFIG_PATH))

print('Config path:', CONFIG_PATH)
print('Symbols:', config.symbols)
print('Date range:', config.start, '->', config.end)
print('Frequency:', config.frequency)
print('Fetch per symbol/week (a):', config.fetch_articles_per_symbol)
print('Combine per prompt (b):', config.combine_articles_per_sample)
print('Holding period days:', config.holding_period_days)
print('Dataset output:', config.dataset_output_path)
print('Backtest output:', config.backtest_output_path)


In [ ]:
# Cell 3 - Fetch/cache Alpaca weekly news and build dataset only
from backtest.alpaca_news_pipeline import run_alpaca_backtest_pipeline

dataset_result = run_alpaca_backtest_pipeline(
    str(CONFIG_PATH),
    run_existing_backtest=False,
)

dataset_result


In [ ]:
# Cell 4 - Inspect cache manifest and a small cache sample
cache_dir = Path(config.cache_dir)
manifest_path = cache_dir / 'cache_manifest.json'
cache_path = cache_dir / 'news_cache.json'

manifest = json.loads(manifest_path.read_text(encoding='utf-8'))
print('Manifest:')
print(json.dumps(manifest, indent=2))

cache_payload = json.loads(cache_path.read_text(encoding='utf-8'))
print('\nCached symbols:', list(cache_payload['articles_by_symbol'].keys())[:5])

first_symbol = next(iter(cache_payload['articles_by_symbol']))
first_windows = cache_payload['articles_by_symbol'][first_symbol]
first_window_key = next(iter(first_windows))
print('\nSample symbol/window:', first_symbol, first_window_key)
print('Articles in sample window:', len(first_windows[first_window_key]))
if first_windows[first_window_key]:
    print(json.dumps(first_windows[first_window_key][0], indent=2)[:1500])


In [ ]:
# Cell 5 - Inspect generated dataset
dataset_path = Path(dataset_result['dataset_path'])
dataset_df = pd.read_csv(dataset_path)

print('Dataset rows:', len(dataset_df))
display(dataset_df.head(10))

print('\nColumns:')
print(dataset_df.columns.tolist())


In [ ]:
# Cell 6 - Weekly dataset summary
summary = (
    dataset_df.groupby(['ticker', 'window_key', 'skip_llm'], dropna=False)
    .size()
    .reset_index(name='n_rows')
)
display(summary.head(30))

print('skip_llm counts:')
print(dataset_df['skip_llm'].value_counts(dropna=False))

print('\npass_reason counts:')
if 'pass_reason' in dataset_df.columns:
    print(dataset_df['pass_reason'].fillna('').value_counts())


In [ ]:
# Cell 7 - Inspect no-signal rows
no_signal_df = dataset_df[
    dataset_df['forced_signal'].fillna('') == 'no_signal'
].copy()

print('No-signal rows:', len(no_signal_df))
display(no_signal_df.head(20))


In [ ]:
# Cell 8 - Run the full weekly Alpaca backtest
# Requires your normal model/runtime env to be configured.
from backtest.alpaca_news_pipeline import run_alpaca_backtest_pipeline

RUN_FULL_BACKTEST = False

if RUN_FULL_BACKTEST:
    backtest_result = run_alpaca_backtest_pipeline(
        str(CONFIG_PATH),
        run_existing_backtest=True,
    )
    print(backtest_result)
else:
    print('Set RUN_FULL_BACKTEST = True to run model inference and backtest.')


In [ ]:
# Cell 9 - Load backtest results and compute metrics
from backtest.backtester import compute_metrics

results_path = Path(config.backtest_output_path)
if results_path.exists():
    results_df = pd.read_csv(results_path)
    print('Backtest rows:', len(results_df))
    metrics = compute_metrics(results_df)
    print(json.dumps(metrics, indent=2, default=str))
    display(results_df.head(10))
else:
    print(f'No backtest results file found yet: {results_path}')


## Notes

- Weekly windows are generated from `config.start` and roll forward in 7-day blocks.
- Each symbol is fetched independently for each weekly window.
- `fetch_articles_per_symbol` is the per-symbol, per-week cap.
- `combine_articles_per_sample` is the number of news items merged into one model prompt.
- Empty weekly windows are written as `skip_llm=True`, `forced_signal=no_signal`, `pass_reason=no_article_provided`.
- The dataset `From ... to ...` range currently uses:
  - `start_date = weekly_end` 之后的下一个美股交易日
  - `end_date = (start_date + holding_period_days)` 之后的下一个美股交易日
